# Desafío · Una fuente cambió sin avisar

La aplicación publicó una versión nueva. El lote contiene un tipo de evento nuevo (`refund`), una columna adicional (`app_version`) y al menos un evento duplicado. Incorporalo sin perder la historia ni duplicar identificadores.

In [0]:
dbutils.widgets.text("student_id", "narela_armani", "Identificador del alumno")
dbutils.widgets.dropdown("scale", "small", ["test", "small", "demo"], "Escala")

In [0]:
%run ../common/config

In [0]:
%run ../common/generate_data

In [0]:
catalog = spark.sql("SELECT current_catalog()").first()[0]
config = CourseConfig(catalog, dbutils.widgets.get("student_id"), dbutils.widgets.get("scale"))
create_course_namespace(spark, config)
challenge_path = f"{config.volume_path}/events_json_v2"
build_challenge_batch(spark, config).coalesce(2).write.mode("overwrite").json(challenge_path)
print(challenge_path)

/Volumes/workspace/bigdata_narela_armani/landing/events_json_v2


## Consigna 1 — Contrato y diagnóstico

Leé el lote nuevo, compará su esquema con `bronze_events` y respondé:

1. ¿Qué campo apareció?
2. ¿Qué valor de `event_type` no existía?
3. ¿Cuántas filas e identificadores distintos contiene?

In [0]:
# TODO: leer challenge_path, inspeccionar el esquema y calcular las tres respuestas.
challenge = spark.read.json(challenge_path)
bronze_events = spark.table(f"{config.catalog}.{config.schema}.bronze_events")

print("--- Esquema del lote nuevo ---")
challenge.printSchema()

nuevas_columnas = set(challenge.columns) - set(bronze_events.columns)
print("Columnas nuevas:", nuevas_columnas)

tipos_nuevos = [r.event_type for r in challenge.select("event_type").distinct().collect()]
tipos_viejos = [r.event_type for r in bronze_events.select("event_type").distinct().collect()]
print("Tipos de evento nuevos:", set(tipos_nuevos) - set(tipos_viejos))

resumen = challenge.agg(
    F.count("*").alias("filas"),
    F.countDistinct("event_id").alias("ids_distintos"),
)
display(resumen)


--- Esquema del lote nuevo ---
root
 |-- app_version: string (nullable = true)
 |-- context: struct (nullable = true)
 |    |-- platform: string (nullable = true)
 |    |-- session_id: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- event_id: long (nullable = true)
 |-- event_ts: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: long (nullable = true)

Columnas nuevas: {'app_version'}
Tipos de evento nuevos: {'refund'}


filas,ids_distintos
10001,10000


## Consigna 2 — Integración controlada

Construí `bronze_events_v2` con estas propiedades:

- Contiene los eventos históricos y el lote nuevo.
- Conserva `app_version`, usando `NULL` para la historia anterior.
- Tiene un único registro por `event_id`.
- Agrega `_source`, `_source_file` y `_ingested_at`.
- Puede volver a ejecutarse sin cambiar el total final.

No modifiques `bronze_events`: queremos conservarlo como evidencia de la primera ingesta.

In [0]:
from pyspark.sql import functions as F

source_table = f"{config.catalog}.{config.schema}.bronze_events"
target_table = f"{config.catalog}.{config.schema}.bronze_events_v2"

def with_ingestion_metadata(frame, source_name, source_path):
    return (frame
        .withColumn("_source", F.lit(source_name))
        .withColumn("_source_file", F.lit(source_path))
        .withColumn("_ingested_at", F.current_timestamp()))

historico = spark.table(source_table)

nuevo = with_ingestion_metadata(
    spark.read.json(challenge_path),
    "events_json_v2",
    challenge_path,
)

unidos = historico.unionByName(nuevo, allowMissingColumns=True)

deduplicado = unidos.dropDuplicates(["event_id"])

(deduplicado.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(target_table))

print(f"{target_table}: {spark.table(target_table).count():,} filas")

workspace.bigdata_narela_armani.bronze_events_v2: 210,000 filas


## Consigna 3 — Verificación automática

Completá las aserciones. Una entrega válida debe demostrar que no hay duplicados y que la columna evolucionada existe.

In [0]:
# TODO: reemplazar los valores pendientes.
result = spark.table(target_table)
total_rows = result.count()
distinct_ids = result.select(F.countDistinct("event_id")).first()[0]
assert total_rows == distinct_ids, "Hay event_id duplicados"
assert "app_version" in result.columns, "No se preservó la evolución del esquema"
assert result.where("event_type = 'refund'").count() > 0, "No se incorporó refund"

## Reflexión final

En no más de 150 palabras, explicá la diferencia entre: detectar una evolución de esquema, aceptarla técnicamente y decidir que es válida para el negocio.

## Respuesta

## Reflexión final

Detectar una evolución de esquema es darse cuenta de que algo cambió. En este caso, al comparar el lote nuevo con `bronze_events` vimos que apareció la columna `app_version` y un tipo de evento nuevo, `refund`. Es solo diagnóstico: todavía no hicimos nada con eso.

Aceptarla técnicamente es lograr que el sistema pueda guardar el cambio sin romperse ni perder historia. Por ejemplo: unir por nombre de columna, dejar `NULL` en `app_version` para los eventos viejos, sacar duplicados y que se pueda volver a correr sin cambiar el total.

Decidir que es válida para el negocio es otra cosa y no la resuelve el código. Alguien tiene que definir qué significa un `refund` (¿resta de las ventas?) y si `app_version` sirve para algo. Que los datos se puedan cargar no quiere decir que estén bien usados.